# Importing Libraries

In [ ]:
from datasets import load_dataset
from transformers import pipeline
import torch

# Check if Cuda is available

In [ ]:
device = 0 if torch.cuda.is_available() else -1

# Load the Dataset

In [ ]:
try:
    dataset = load_dataset("DynamicSuperb/Sentiment_Analysis_SLUE-VoxCeleb", split="test")
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"Failed to load dataset: {e}")
    exit()

# Load the Models

In [ ]:
asr_models = [
    "AventIQ-AI/whisper-audio-to-text",
    "facebook/s2t-small-librispeech-asr"
]

In [ ]:
sentiment_analysis_models = [
    "siebert/sentiment-roberta-large-english",
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "tabularisai/multilingual-sentiment-analysis"
]

## Testing every combination of model

In [ ]:
for asr_model_name in asr_models:
    print(f"\n{'='*50}")
    print(f"--- Initializing ASR Model: {asr_model_name} ---")
    print(f"{'='*50}")
    
    try:
        # Initialize the ASR pipeline with the current model
        asr = pipeline(
            "automatic-speech-recognition",
            model=asr_model_name,
            device=device
        )
        print(f"Successfully initialized ASR model: {asr_model_name}")
    except Exception as e:
        print(f"Failed to initialize ASR model {asr_model_name}: {e}")
        # Skip to the next ASR model if initialization fails
        continue

    # Iterate over each sentiment analysis model for the current ASR model
    for sentiment_model_name in sentiment_analysis_models:
        print(f"\n{'-'*50}")
        print(f"--- Testing with Sentiment Model: {sentiment_model_name} ---")
        print(f"{'-'*50}")

        try:
            # Initialize the sentiment analysis pipeline with the current model
            sentiment_analyzer = pipeline(
                "sentiment-analysis",
                model=sentiment_model_name,
                device=device
            )
            print(f"Successfully initialized sentiment analysis model: {sentiment_model_name}")
        except Exception as e:
            print(f"Failed to initialize sentiment analysis model {sentiment_model_name}: {e}")
            # Skip to the next sentiment model if initialization fails
            continue

        # Process a small subset of the dataset to test the pipeline
        for i, example in enumerate(dataset.select(range(2))):
            print(f"\n--- Processing Example {i+1} ---")

            # Extract the audio array from the example
            audio_input = example["audio"]["array"]
            
            ground_truth_sentiment = example["label"]

            print(f"Ground-Truth Sentiment: {ground_truth_sentiment}")

            # Perform ASR on the audio input
            transcription_result = asr(audio_input)
            transcribed_text = transcription_result["text"]
            print(f"Transcribed Text (from {asr_model_name}): {transcribed_text}")

            # Perform sentiment analysis if the transcription is not empty
            if transcribed_text:
                sentiment_result = sentiment_analyzer(transcribed_text)
                print(f"Predicted Sentiment (from {sentiment_model_name}): {sentiment_result[0]['label']} (Score: {sentiment_result[0]['score']:.4f})")
            else:
                print("Predicted Sentiment (from Text): Could not be determined (empty transcription).")
                
print("\nAll model combinations have been tested.")